In [1]:
import sys
import os

# Add parent directory to path to import from app
sys.path.insert(0, os.path.abspath('..'))

from app.db.core.models.market_data_models import *
from app.db.core.models.user_data_models import *
from app.db.core.models.prophit_alts_models import *
from app.db.core.db_config import UserSession, MarketSession, ProphitAltsSession
from app.utils.decorators.database import with_session, with_transaction
from app.utils.serialize_output import serialize_sqlalchemy_obj
import json
import pandas as pd
from app.core.calculations.core.helpers import build_returns_df_for_dates
from datetime import datetime, timedelta
import requests
import os
from dotenv import load_dotenv
from sklearn.decomposition import PCA
from app.utils.gpt_parser import canonical_portfolio


In [2]:
x = [
  {
    "ticker": "BJ",
    "position": "long",
    "thesis": "BJ’s Wholesale Club is a membership-driven retail model leveraging private label economics and operational efficiencies. Strong ROE (7.2-7.6%), steady operating margins (~4%) and industry-leading free cash flow conversion underpin financial durability. Its high member renewal rates, historical inflation resilience, and focus on value positioning encourage robust customer traffic despite sector headwinds. Membership renewal and inflation hedging further reinforce downside risk protection.",
    "key_drivers": "1) Resilient membership economics and value focus; 2) Outperformance in both pro-cyclical and recessionary periods due to club model; 3) Consistent margin and FCF generation; 4) High private label mix; 5) Operational scale and cost control; 6) Defensive sector exposure with low beta (0.14); 7) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "CAG",
    "position": "long",
    "thesis": "Conagra offers contrarian value at a discount to peers, supported by visible operating margin recovery and sequential improvements from pricing/cost actions. The company has low leverage and is trading at PE multiples that reflect cyclical troughs. Defensive SKU breadth and strong brand moat provide stability during volatile packaging and input cycles.",
    "key_drivers": "1) Improving sequential margin/ROIC; 2) Attractive valuation (PE ~10-21); 3) Margins recovering from inflation pass-through; 4) Strong product brand/retail relationships; 5) Broad portfolio with stable consumer staples demand; 6) Defensive low beta (0.07); 7) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.05
  },
  {
    "ticker": "CCEP",
    "position": "long",
    "thesis": "Coca-Cola Europacific Partners, the premier KO bottler, excels in shelf space, operating efficiency, and pricing power. High gross and operating margins, significant scale, and regional/localization strategies drive high quality compounder status. Strong track record of low volatility and defensive returns.",
    "key_drivers": "1) Premium bottling/distribution rights; 2) Above-average margin and FCF yield; 3) Deep pricing power; 4) Risk mitigation via scale and shelf presence; 5) Advantageous localization/shelf optimization strategies; 6) Defensive, low beta (0.30); 7) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "BG",
    "position": "long",
    "thesis": "Bunge is a leading agricultural processor/originator trading at a discount to normalized mid-cycle economics. Robust balance sheet, strategic positioning in global commodity flows, and effective risk management drive strong normalized FCF. Positive recent momentum and effective capital returns support upside potential.",
    "key_drivers": "1) Cash generator with proven hedging strategies; 2) Underappreciated normalized earnings power; 3) Conservative capital returns (buyback/dividend); 4) Attractive value (PE ~7.7-12.7); 5) Good defensive beta (0.39); 6) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.053
  },
  {
    "ticker": "CL",
    "position": "long",
    "thesis": "Colgate-Palmolive offers a durable compounder profile with industry-leading gross margin (>35%), strong market share in oral care, consistent FCF, and modest leverage. Entry after a pullback offers an attractive combination of defensive quality and potential upside from GM expansion.",
    "key_drivers": "1) Dominant brand in oral care; 2) Stable, high margin model; 3) FCF compounding; 4) Defensive sector characteristics; 5) Low to moderate operating risk; 6) Support: get_ticker_fundamental_data.",
    "allocation": 0.052
  },
  {
    "ticker": "CHD",
    "position": "short",
    "thesis": "Church & Dwight faces aging brand risks, rising input costs, and promotional spending pressures—all at a premium relative to slower growth prospects. Recent performance is poor and profitability is susceptible to further compression, while the current share price discounts a return to previous margin highs.",
    "key_drivers": "1) Weak risk-adjusted returns; 2) Premium valuation relative to sector; 3) Margin headwinds from promo intensity; 4) Regulatory cost exposure; 5) Negative alpha; 6) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "CELH",
    "position": "short",
    "thesis": "Celsius Holdings’ astronomical valuation and excessive volatility (beta ~1.0, volatility 64%) present large risks as growth decelerates. Recent rating downgrades and shelf reset risk undermine confidence in continued momentum. High embedded growth expectations, plus competition and de-rating risk, skew risk/reward sharply negative.",
    "key_drivers": "1) High price/book, PE, and volatility; 2) Growth normalization; 3) Increasing competition; 4) Ratings/target cut cycle; 5) Sensitivity to shelf resets/promos; 6) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "COCO",
    "position": "short",
    "thesis": "The Coca-Cola Company faces mean reversion risk after outsized momentum. Valuation is stretched (high PE/PB), and shelf/nearly saturated category/retailer private label threat poses downside risk. High beta and volatility compound downside on earnings disappointment or sector rotation.",
    "key_drivers": "1) Rich valuation; 2) Rising competitive risk; 3) Retailer shelf reset incentive; 4) High price momentum at risk of reversal; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "DG",
    "position": "long",
    "thesis": "Dollar General is a trade-down beneficiary with attractive value (PE 12-14) and a visible margin recovery plan. Management’s supply chain/operating improvements are expected to drive margin repair into 2026. While negative beta remains a concern, DG provides unique factor diversification in a low-beta portfolio.",
    "key_drivers": "1) Trade-down/resilient demand in value retail; 2) Self-help margin initiatives; 3) Attractive forward valuation; 4) Low correlation to consumer cyclicality; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "CLX",
    "position": "short",
    "thesis": "Clorox bears the dual burden of excessive leverage (debt/equity 9-110x) and negative returns. Its weak ROE/ROA and continual margin fragility present compelling grounds for a short, particularly as pandemic windfalls fade.",
    "key_drivers": "1) High leverage; 2) Weak operational efficiency; 3) Deteriorating fundamentals; 4) Recent negative momentum; 5) High beta (0.26); 6) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "COST",
    "position": "short",
    "thesis": "Costco’s stellar operations are fully priced in, with best-in-class operational track record offset by staggeringly high PE (59-66) and P/B (16-18). Valuation is unsustainable should the business see even temporary slowing or margin reversion.",
    "key_drivers": "1) Highest valuation risk in staples space; 2) Beta 0.62; 3) Modest margin benefits fading; 4) Macro headwinds not reflected in price; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "COTY",
    "position": "short",
    "thesis": "Coty’s consistent net losses, highly negative momentum, and high beta/volatility with minimal balance sheet resilience make it an ideal short. Downside is supported by poor quality, and inability to turn portfolio rationalization into returns.",
    "key_drivers": "1) Consistent net losses and negative cash flow; 2) High downside volatility beta (1.05); 3) Negative 1Y and 3Y returns; 4) Margin erosion; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "EPC",
    "position": "long",
    "thesis": "Edgewell Personal Care is a rare value in personal care (PE 9.6-13, P/B <1) with steadily improving margins via self-help/brand momentum (Sun, Hawaiian Tropic, and international). Despite sector headwinds, recovery in ROIC and balance sheet supports exposure for staple diversification.",
    "key_drivers": "1) Valuation discount; 2) Self-help and productivity program execution; 3) Improving gross/operating margins (42-44%); 4) Defensive FCF profile; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.05
  },
  {
    "ticker": "FIZZ",
    "position": "short",
    "thesis": "National Beverage Corp. is a high-multiple, premium beverage with modest moat and brand power. Strong past ROIC but downtrending 1Y returns, negative alpha, and overvaluation justify a short for mean reversion risk and core staple factor exposure.",
    "key_drivers": "1) Downside momentum (-16% 1Y total return); 2) Overstretched multiples (PE ~19-22); 3) Brand/volume risk; 4) Beta 0.22; 5) Limited scale outside brand; 6) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "FRPT",
    "position": "short",
    "thesis": "Freshpet presents an unattractive risk/reward following severe drawdown (-64% 1Y return), high volatility, and mean reversion risk. Current multiples and business swings are not justified by fundamentals.",
    "key_drivers": "1) Very high valuation; 2) Negative 1Y and 3Y returns; 3) High volatility; 4) Category risk; 5) Unstable margin performance; 6) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "GIS",
    "position": "long",
    "thesis": "General Mills brings stability and margin resilience to the portfolio. Although momentum is out of favor, their low beta, recurring FCF, and robust balance sheet enhance overall portfolio defensiveness.",
    "key_drivers": "1) Margin and cash flow stability; 2) Broad product mix; 3) Defensive sector exposure; 4) Low beta (0.01); 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "HSY",
    "position": "short",
    "thesis": "Hershey faces margin compression from cocoa spikes (2024-25), rising cost complexity, and softening volume. Despite stability, deteriorating profitability and high valuation (PE up to 134x) support a short position ahead of cyclical headwinds.",
    "key_drivers": "1) Margin compression drivers; 2) Deteriorating trend in profit/loss; 3) Cocoa cost risk; 4) Elevated sector multiples; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "INGR",
    "position": "long",
    "thesis": "Ingredion offers a cash-generative, resilient growth profile (PE ~11), with considerable margin and pricing power. Regulatory tailwinds support ingredient and specialty exposure. Stable positive alpha and moderate beta ensure robust, defensive long positioning.",
    "key_drivers": "1) Stable revenue and margin mix (operating ~15%); 2) Moderate leverage, solid cash flow; 3) Regulatory tailwind (EUDR/FSMA); 4) Low beta (0.37); 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "IPAR",
    "position": "long",
    "thesis": "Inter Parfums exhibits quality compounder characteristics in prestige fragrance, with an expanding license base, growing international platform, and margin compounding via asset-light execution. Strong brand relationships enable pricing power but, near-term momentum is challenged. Appropriate for modest allocation.",
    "key_drivers": "1) Pricing power; 2) Tariff and risk mitigation strategies; 3) Diversified brand pipeline; 4) FCF support for continued growth; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "KDP",
    "position": "long",
    "thesis": "Keurig Dr Pepper is a large-scale, diversified beverage franchise stabilizing from a cyclical trough. Margin normalization path and moderate beta (0.15) make for a defensive, low-vol risk allocation.",
    "key_drivers": "1) Underappreciated mean reversion in margins; 2) Diversified product exposure; 3) Margin of safety in defensive beverage positioning; 4) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "KLG",
    "position": "short",
    "thesis": "Kellogg faces headwinds from portfolio mix, pricing power erosion, and competitive threat from private label. Volatile returns, high price/book (3.9-5.1), and low profitability increase mean reversion risk.",
    "key_drivers": "1) Portfolio pressure from value and private label; 2) Negative return trend; 3) High beta risk; 4) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "KMB",
    "position": "long",
    "thesis": "Kimberly-Clark offers robust margin and cash flow defensiveness for the staples sector. Strong historical outperformance, low beta (0.13), and a thick FCF cushion make KMB a cornerstone defensive allocation.",
    "key_drivers": "1) High cash flow conversion; 2) Positive 3Y return trend; 3) Low/defensive sector beta; 4) Recurring product demand; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "KO",
    "position": "long",
    "thesis": "Coca-Cola’s global beverage franchise is the premier low-volatility core holding for staples, benefiting from diversified global operations, durable pricing power, high margin/FCF, and unmatched sector resilience.",
    "key_drivers": "1) Superior shelf and pricing power; 2) Consistent margin expansion; 3) Defensive, low beta (0.09); 4) ESG-driven packaging strategies; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "MO",
    "position": "long",
    "thesis": "Altria sits at a valuation trough with resilient pricing and best-in-class cash conversion. Menthol regulatory risks and nicotine caps are distant. Recent momentum improvement, high yield, and low beta make MO a core quality/defensive long.",
    "key_drivers": "1) Defensive sector leader in U.S. comb tobacco; 2) Strong dividend and cash return; 3) Low beta, high Sharpe and alpha; 4) Regulatory risk over-discounted; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "OLPX",
    "position": "short",
    "thesis": "Olaplex’s volatility, persistent negative returns, and high beta (1.44) reflect deteriorating operational performance and multiple expansion risk. No sign of a floor on margin/brand protection.",
    "key_drivers": "1) High beta and volatility; 2) Downward price/margin momentum; 3) Litigation/strategy overhang; 4) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "REYN",
    "position": "long",
    "thesis": "Reynolds Consumer Products is a defensive FCF-rich holding in household packaging. Despite weak recent returns and only modest operating margins, the low beta (0.24) and business defensiveness make it a core sector holding.",
    "key_drivers": "1) Defensive sector exposure; 2) Reasonable FCF and operational leverage; 3) Recurring packaging demand; 4) Stable balance sheet; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "RLX",
    "position": "long",
    "thesis": "RLX Technology offers deep value in Chinese ENDS with immense net cash cushion, high net margin, and above-market alpha. Risk diversification and strong 1Y returns recommend tactical portfolio inclusion.",
    "key_drivers": "1) Deep value metrics; 2) High net margin; 3) Exceptional balance sheet; 4) Strong 1Y return and alpha; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "SFM",
    "position": "short",
    "thesis": "Sprouts Farmers Market faces significant trade-down/valuation risk, high leverage, and negative returns. Beta of 0.63 and past mean reversion after pandemic. A credible short for risk management.",
    "key_drivers": "1) Elevated leverage; 2) Subpar 1Y trending total/share returns; 3) Vulnerable at upper valuation; 4) Margin compression risk; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "TAP",
    "position": "long",
    "thesis": "Molson Coors is a deep-value brewer benefiting from earnings/OCF recognition inflection and sector negativity. With low beta (~0.38), a positive margin recovery path, and brand moat, TAP is positioned for both defensive yield and multiples expansion.",
    "key_drivers": "1) Inflecting margin trajectory; 2) Deep value; 3) Defensive beverage basket; 4) Brand and shelf moat; 5) Low beta; 6) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.052
  },
  {
    "ticker": "STZ",
    "position": "short",
    "thesis": "Constellation Brands faces high leverage, negative 1Y returns (-40%), and volatility—despite top brands, margin performance is severely at risk. Short captures mean reversion and risk premium.",
    "key_drivers": "1) High leverage; 2) Negative recent returns; 3) Margin and volume contraction risk; 4) Sector/consumption volatility; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  },
  {
    "ticker": "TGT",
    "position": "short",
    "thesis": "Target faces valuation risk, macro and retail execution uncertainty, and negative price momentum. Beta (0.86) and significant 1Y drawdown compound the risk. Short is a retail/momentum hedge in the staples short book.",
    "key_drivers": "1) Macro and retail sector risk; 2) Beta diversification; 3) Execution risk; 4) Negative pricing momentum; 5) Support: get_ticker_fundamental_data, get_ticker_performance_and_risk.",
    "allocation": 0.04
  }
]
portfolio = canonical_portfolio(x)

In [ ]:
start_date = datetime.now() - timedelta(days=365*2)
end_date = datetime.now()
print(portfolio)

tickers = []
for ticker in portfolio.keys():
    tickers.append(ticker)

df = build_returns_df_for_dates(tickers, start_date, end_date, drop_rows='none')
print(df.head())
print(df.tail())

{'BJ': {'allocation': 0.052, 'position': 'long'}, 'CAG': {'allocation': 0.05, 'position': 'long'}, 'CCEP': {'allocation': 0.052, 'position': 'long'}, 'BG': {'allocation': 0.053, 'position': 'long'}, 'CL': {'allocation': 0.052, 'position': 'long'}, 'CHD': {'allocation': 0.04, 'position': 'short'}, 'CELH': {'allocation': 0.04, 'position': 'short'}, 'COCO': {'allocation': 0.04, 'position': 'short'}, 'DG': {'allocation': 0.052, 'position': 'long'}, 'CLX': {'allocation': 0.04, 'position': 'short'}, 'COST': {'allocation': 0.04, 'position': 'short'}, 'COTY': {'allocation': 0.04, 'position': 'short'}, 'EPC': {'allocation': 0.05, 'position': 'long'}, 'FIZZ': {'allocation': 0.04, 'position': 'short'}, 'FRPT': {'allocation': 0.04, 'position': 'short'}, 'GIS': {'allocation': 0.052, 'position': 'long'}, 'HSY': {'allocation': 0.04, 'position': 'short'}, 'INGR': {'allocation': 0.052, 'position': 'long'}, 'IPAR': {'allocation': 0.052, 'position': 'long'}, 'KDP': {'allocation': 0.052, 'position': 'long

In [10]:
# Push the DataFrame `df` to an Excel sheet
excel_filename = "long_only_returns.xlsx"
df.to_excel(excel_filename, index=True)
print(f"DataFrame exported to {excel_filename}")


DataFrame exported to long_only_returns.xlsx
